# 9주차 — 생성형 AI API 첫걸음
### 인공지능 플랫폼실습 · 인공지능소프트웨어과

---

## 이번 주 학습 목표
1. API 가 무엇이고 3~5주차의 "모델 내려받기"와 무엇이 다른지 설명할 수 있다.
2. Gemini API 키를 발급받아 Colab 보안 비밀에 안전하게 저장할 수 있다.
3. 파이썬 코드로 생성형 AI API 를 호출하고 응답을 출력할 수 있다.
4. 토큰과 요금 구조를 이해하고, 무료 등급의 보안 수칙을 지킬 수 있다.

## 3~5주차와 무엇이 다른가
```
3~5주차 : 모델 파일을 내 Colab 으로 내려받아 실행     -> 모델이 작아야 함, 인터넷 없이도 계산은 내 쪽
9주차   : 질문만 구글 서버로 보내고 답을 받아 옴      -> 아주 큰 모델도 쓸 수 있음, 대신 키 · 한도 · 요금이 생김
```

## 준비물
| 항목 | 조건 |
|---|---|
| Google 계정 | 본인 계정 (학교 계정이 막혀 있으면 개인 계정) |
| 나이 | **만 18세 이상** (Gemini API 약관) |
| 결제 수단 | **필요 없음 — 등록하지 않음** |

> ⏱ 예상 소요 시간: 약 125분
> 검증 기준 — google-genai 2.12.1 (Colab 기본 google-auth 2.49.0 과 충돌 없음) · 모델 `gemini-3.5-flash-lite` · Interactions API (2026-09 공식 문서 기준)

---
# STEP 0. 설치와 [점검]  (약 5분)

In [5]:
# ============================================================
# [실습 0-1] 설치
# ------------------------------------------------------------
# google-genai : 구글이 만든 Gemini API 파이썬 도구
# 버전을 2.12.1 로 고정합니다.
#  · 2.13 이후 버전은 google-auth 를 새 버전으로 바꿔 버려서,
#    Colab 이 쓰는 google-auth 2.49.0 과 충돌합니다 (빨간 ERROR 문장이 나옴).
#  · 2.12.1 에도 오늘 쓰는 Interactions API 가 들어 있습니다.
# 이미 빨간 ERROR 를 본 경우 : [런타임] → [런타임 연결 해제 및 삭제] 후 처음부터 다시 실행
# ============================================================

!pip install -q google-genai==2.12.1

In [6]:
# ============================================================
# [점검] 한 번만 실행해서 결과를 확인합니다
# ------------------------------------------------------------
# 막혔을 때 이 셀의 출력을 교수에게 보여 주세요.
# ★ 키 값 자체는 절대 화면에 찍지 않습니다. 읽혔는지만 확인합니다.
# ============================================================

import google.genai
from google import genai

print("google-genai 버전 :", google.genai.__version__)

import google.auth
print("google-auth 버전  :", google.auth.__version__, "(Colab 기본 2.49.0 이면 정상)")

check_client = genai.Client(api_key="check-only")          # 기능 확인용 (호출하지 않음)
print("Interactions API  :", "있음 ✅" if hasattr(check_client, "interactions") else "없음 ❌ -> 설치 셀을 다시 실행")

from google.colab import userdata
try:
    key_text = userdata.get("GEMINI_API_KEY")
    print("보안 비밀 키      : 읽기 성공 ✅ (글자 수", len(key_text), ")")
except Exception as error:
    print("보안 비밀 키      : 아직 없음 ⏳ (STEP 2 를 마치면 다시 실행) —", type(error).__name__)

google-genai 버전 : 2.12.1
google-auth 버전  : 2.49.0 (Colab 기본 2.49.0 이면 정상)
Interactions API  : 있음 ✅
보안 비밀 키      : 읽기 성공 ✅ (글자 수 53 )


---
# STEP 1. API 란 무엇인가  (약 10분)

**API(Application Programming Interface)는 다른 프로그램의 기능을 정해진 형식의 요청으로 빌려 쓰게 해 주는 창구입니다.**
생성형 AI API 는 구글 · 오픈AI 같은 회사의 서버에 있는 큰 모델에게 질문(요청)을 보내고 답(응답)을 받아 오는 창구입니다.

```
내 Colab 코드 ──(요청: 모델 이름 + 질문 + API 키)──▶  구글 서버의 Gemini 모델
              ◀──(응답: 답 글 + 사용한 토큰 수)───────
```

비유하면 식당의 주문 창구와 같습니다. 주방(모델)에 직접 들어가지 않고, 정해진 주문서(요청 형식)를 내면 음식(응답)이 나옵니다.

| 용어 | 뜻 |
|---|---|
| 요청(Request) | 내 코드가 서버로 보내는 것 — 모델 이름 · 질문 · 옵션 |
| 응답(Response) | 서버가 돌려주는 것 — 답 글 · 토큰 수 · 상태 |
| API 키 | 누가 요청했는지 알려 주는 비밀번호. 한도와 요금이 이 키에 붙음 |
| 토큰(Token) | 모델이 글을 자르는 단위. 한도와 요금을 세는 기준 |
| 무료 등급 | 결제 수단 없이 쓰는 단계. 대신 한도가 있고 입력이 제품 개선에 쓰임 |

---
# STEP 2. API 키 발급과 저장  (약 15분)

### [실습 2-1] 키 발급 — Google AI Studio
| 순서 | 할 일 |
|---|---|
| 1 | `aistudio.google.com` 접속 → 본인 Google 계정으로 로그인 |
| 2 | 처음이면 약관 동의 화면 → 내용 확인 후 동의 |
| 3 | 왼쪽 메뉴의 **API Keys(API 키)** 페이지 |
| 4 | **Create API key(API 키 만들기)** → 프로젝트는 **결제가 연결되지 않은 것**(없으면 새로 만들기) → 만들어진 키 **복사** |

> ★ **결제(Billing) 설정은 누르지 않습니다.** 결제 계정을 연결하지 않으면 무료 등급으로만 동작해 요금이 나오지 않습니다.
> ★ 결제가 연결된 프로젝트(예: Google Cloud 무료 체험에 쓰던 프로젝트)에서 키를 만들면 **402 오류**(`Your prepayment credits are depleted`)가 납니다 → AI Studio 에서 **새 프로젝트**를 만들어 키를 다시 발급합니다. 크레딧을 충전하지 않습니다.
> ★ 2026-05-28 부터 AI Studio 에서 새로 만든 키는 **auth key** 로 만들어지며, 유출이 감지되면 빠르게 사용이 막힙니다. (공식 문서)

### [실습 2-2] Colab 보안 비밀에 저장
| 순서 | 할 일 |
|---|---|
| 1 | Colab 왼쪽 **🔑(보안 비밀)** 아이콘 |
| 2 | **새 보안 비밀 추가** |
| 3 | 이름 : `GEMINI_API_KEY` (띄어쓰기 없이 정확히) |
| 4 | 값 : 복사한 키 붙여넣기 |
| 5 | **노트북 액세스** 스위치 켜기 |

> 보안 비밀에 넣은 키는 노트북을 공유해도 다른 사람에게 보이지 않습니다. **코드 안에 키를 직접 적지 않습니다.**

In [7]:
# ============================================================
# [실습 2-3] 보안 비밀에서 키 읽기
# ------------------------------------------------------------
# userdata.get("이름") 은 Colab 보안 비밀에 저장한 값을 글자로 돌려주는 함수입니다.
# 실패하면 이름 철자와 "노트북 액세스" 스위치를 확인하세요.
# ============================================================

from google.colab import userdata

try:
    api_key = userdata.get("GEMINI_API_KEY")
    print("✅ 키를 읽었습니다. (키 값은 화면에 찍지 않습니다)")
except Exception as error:
    api_key = None
    print("❌ 키를 읽지 못했습니다 :", type(error).__name__)
    print("   · 보안 비밀 이름이 GEMINI_API_KEY 인지")
    print("   · 노트북 액세스 스위치가 켜져 있는지 확인하세요")

✅ 키를 읽었습니다. (키 값은 화면에 찍지 않습니다)


---
# STEP 3. 첫 API 호출  (약 20분)

오늘은 구글이 **2026년 6월 정식 출시한 Interactions API** 로 호출합니다.
공식 문서는 새 프로젝트에 이 방식을 권하고, 예전 방식(`generate_content`)은 legacy(계속 지원)로 분류합니다.

모델은 **`gemini-3.5-flash-lite`** 를 씁니다. 무료 한도가 **분당 15회 · 하루 500회**로 넉넉하기 때문입니다. (최신 `gemini-3.8-flash` 는 하루 20회뿐 — STEP 5 표 참고)

In [8]:
# ============================================================
# [실습 3-1] 첫 질문 보내기
# ------------------------------------------------------------
# client          : Gemini 서버와 연결해 주는 통로 (반드시 변수에 담아 씀)
# http_options    : 기다리는 시간 · 다시 시도 규칙 (없으면 한도 초과 때 말없이 몇 분씩 기다림)
# MODEL_NAME      : 쓸 모델 이름. 모델이 바뀌면 이 한 줄만 고침
# interactions.create(model=..., input=...) : 질문을 보내고 답을 받는 함수
# .output_text    : 응답에서 답 글만 꺼낸 것
# ============================================================

from google import genai
from google.genai import types

client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(
        timeout=120_000,                                              # 2분(120,000ms) 안에 답이 없으면 멈추고 오류를 냄
        retry_options=types.HttpRetryOptions(http_status_codes=[500, 503]),  # 429(한도 초과)는 몰래 기다리지 않고 바로 알려 줌
    ),
)
MODEL_NAME = "gemini-3.5-flash-lite"         # 무료 한도 : 분당 15회 · 하루 500회 (3.8 Flash 는 하루 20회뿐)

interaction = client.interactions.create(
    model=MODEL_NAME,
    input="인공지능이 무엇인지 고등학생에게 두 문장으로 설명해 줘.",
)

print(interaction.output_text)

인공지능이란 인간의 학습하고 추론하는 지능적인 능력을 컴퓨터나 소프트웨어를 통해 구현한 기술입니다. 이는 방대한 데이터를 스스로 분석하고 학습하여, 사람처럼 판단하고 문제를 해결할 수 있게 해줍니다.


> ### 결과가 이렇게 나오면 성공입니다
> 두 문장 안팎의 한국어 설명이 찍힙니다. 문장은 실행할 때마다 조금씩 달라집니다.
> (생성형 AI 는 같은 질문에도 매번 새로 글을 만들기 때문입니다)

In [9]:
# ============================================================
# [실습 3-2] 응답 뜯어보기 — 답 글 말고도 들어 있는 것
# ------------------------------------------------------------
# usage : 이번 요청에 쓴 토큰 수 (한도와 요금을 세는 기준)
# ============================================================

print("응답 번호(id)   :", interaction.id)
print("상태(status)    :", interaction.status)
print("모델            :", interaction.model)
print("입력 토큰       :", interaction.usage.total_input_tokens)
print("출력 토큰       :", interaction.usage.total_output_tokens)
print("생각 토큰       :", interaction.usage.total_thought_tokens)
print("전체 토큰       :", interaction.usage.total_tokens)

응답 번호(id)   : v1_ChdCOXZPYXZlRUJmbkctOFlQc2FTTXFBaxIXQjl2T2F2ZUVCZm5HLThZUHNhU01xQWs
상태(status)    : completed
모델            : gemini-3.5-flash-lite
입력 토큰       : 21
출력 토큰       : 50
생각 토큰       : 0
전체 토큰       : 71


> **생각 토큰(thought tokens)** 은 모델이 답을 쓰기 전에 속으로 따져 본 양입니다.
> 화면에는 안 나오지만 토큰으로는 세어집니다. `None` 이 찍히면 이번에는 생각 단계가 없었다는 뜻입니다.

In [10]:
# ============================================================
# [실습 3-3] 질문을 함수로 만들기 — 여러 번 편하게 묻기
# ============================================================

def ask_gemini(question):
    """질문을 보내고 답 글을 돌려준다"""
    result = client.interactions.create(model=MODEL_NAME, input=question)
    return result.output_text


questions = [
    "파이썬에서 리스트와 튜플의 차이를 한 문장으로 알려 줘.",
    "API 키를 코드에 직접 적으면 왜 위험한지 한 문장으로 알려 줘.",
    "오늘 저녁 메뉴를 하나만 추천해 줘.",
]

for number, question in enumerate(questions, start=1):
    print(f"[{number}] {question}")
    print("   →", ask_gemini(question))
    print()

[1] 파이썬에서 리스트와 튜플의 차이를 한 문장으로 알려 줘.
   → 리스트는 생성 후 그 내용을 변경할 수 있는 **가변(Mutable)** 객체이고, 튜플은 변경할 수 없는 **불변(Immutable)** 객체입니다.

[2] API 키를 코드에 직접 적으면 왜 위험한지 한 문장으로 알려 줘.
   → API 키를 코드에 직접 적으면, 코드 저장소(GitHub 등)가 해킹되거나 공개되었을 때 누구나 키를 탈취해 악용할 수 있어 **금전적 손해나 보안 사고로 이어질 수 있기 때문**입니다.

[3] 오늘 저녁 메뉴를 하나만 추천해 줘.
   → 오늘 저녁은 따뜻하고 얼큰한 **'김치찌개'** 어떠세요? 

돼지고기 듬뿍 넣고 잘 익은 김치로 푹 끓여내면, 다른 반찬 없이도 밥 한 공기 뚝딱 비우기 딱 좋죠. 

오늘 저녁 메뉴로 강력 추천합니다! 맛있게 드세요!



---
# STEP 4. 옵션 바꿔 보기  (약 15분)

`generation_config` 는 답을 만드는 방식을 정하는 옵션 묶음입니다.

| 옵션 | 뜻 |
|---|---|
| `max_output_tokens` | 답의 최대 길이 (토큰 수) |
| `thinking_level` | 답하기 전에 얼마나 생각할지 — `"minimal"`(기본값) · `"low"` · `"medium"` · `"high"` |

In [11]:
# ============================================================
# [실습 4-1] 답 길이 제한 — max_output_tokens
# ------------------------------------------------------------
# 같은 질문을 짧게 / 길게 제한해서 비교합니다.
# 너무 짧게 잡으면 문장이 중간에 끊길 수 있습니다.
# ============================================================

question = "대한민국의 수도를 소개해 줘."

for limit in [30, 300]:
    result = client.interactions.create(
        model=MODEL_NAME,
        input=question,
        generation_config={"max_output_tokens": limit},
    )
    print(f"--- max_output_tokens = {limit} (실제 출력 토큰 {result.usage.total_output_tokens})")
    print(result.output_text)
    print()

--- max_output_tokens = 30 (실제 출력 토큰 26)
대한민국의 수도는 **서울특별시(Seoul)**입니다. 

서울은 한반도의 중서부에 위치하며

--- max_output_tokens = 300 (실제 출력 토큰 296)
대한민국의 수도는 **서울특별시(Seoul)**입니다. 

서울은 한반도의 중부, 한강을 끼고 자리 잡은 대한민국의 정치, 경제, 사회, 문화의 중심지입니다. 서울을 몇 가지 핵심 키워드로 소개해 드릴게요.

### 1. 600년의 역사와 현대가 공존하는 도시
서울은 조선 시대(1392~1910)부터 지금까지 약 600년 동안 대한민국의 중심지였습니다. 
* **역사적 명소:** 조선 시대 왕궁인 **경복궁, 창덕궁**을 비롯해 전통 가옥이 모여 있는 **북촌 한옥마을**, 서울을 둘러싼 **서울성곽** 등이 도심 속에 고스란히 남아 있습니다.
* **현대적 명소:** 세계적인 초고층 빌딩인 **롯데월드타워**, 동대문디자인플라자(DDP), 그리고 트렌디한 카페와 상점이 많은 성수동, 홍대 등이 조화를 이루고 있습니다.

### 2. 교통과 인프라
* **세계 최고 수준의 대중교통:** 지하철과 버스 네트워크가 매우 체계적이고 편리하게 잘 갖추어져 있습니다.
* **스마트 시티:** 초고속 인터넷망과 첨단 디지털 기술이 일상화



In [12]:
# ============================================================
# [실습 4-2] 생각 수준 — thinking_level
# ------------------------------------------------------------
# 생각을 많이 할수록 어려운 문제에 강하지만, 시간과 토큰이 더 듭니다.
# time.perf_counter() 는 걸린 시간을 재는 초시계입니다.
# ============================================================

import time

puzzle = "사과 3개가 있는데 2개를 먹고 5개를 더 샀어. 그중 절반을 친구에게 주면 몇 개가 남아?"

for level in ["minimal", "high"]:
    start = time.perf_counter()
    result = client.interactions.create(
        model=MODEL_NAME,
        input=puzzle,
        generation_config={"thinking_level": level},
    )
    seconds = time.perf_counter() - start
    print(f"--- thinking_level = {level} · {seconds:.1f}초 · 생각 토큰 {result.usage.total_thought_tokens}")
    print(result.output_text)
    print()

--- thinking_level = minimal · 65.7초 · 생각 토큰 0
현재 남아 있는 사과는 **3개**야. 

계산해 볼까?

1. 처음에 사과 3개가 있었어. ($3$)
2. 2개를 먹었으니까 $3 - 2 = 1$개 남았지.
3. 거기에 5개를 더 샀어. $1 + 5 = **6개**$가 되었어.
4. 그중 절반을 친구에게 주었으니, 남은 것도 절반이겠지? $6 \div 2 = **3개**$야.

--- thinking_level = high · 49.2초 · 생각 토큰 469
정답은 **3개**입니다! 

과정을 차근차근 계산해 볼게요.

1. 처음에 사과가 **3개** 있었어요.
2. 2개를 먹었으니 **1개**가 남아요. (3 - 2 = 1)
3. 거기에 5개를 더 샀으니 총 **6개**가 됩니다. (1 + 5 = 6)
4. 그중 절반(6의 절반인 3개)을 친구에게 주었으니, 남은 사과는 **3개**입니다. (6 ÷ 2 = 3)



---
# STEP 5. 토큰과 요금  (약 15분)

**무료 등급은 결제 수단이 없으므로 요금이 나오지 않습니다.** 그래도 요금 구조를 알아 두면 한도가 왜 있는지, 실제 서비스에 얼마가 드는지 알 수 있습니다.

### gemini-3.5-flash-lite 의 유료 등급 가격 (공식 요금표 · 100만 토큰당)
| 구분 | 가격 |
|---|---|
| 입력 | $0.30 |
| 출력 (생각 토큰 포함) | $2.50 |

### 무료 등급 한도 (2026-10-05 AI Studio 화면 기준)
| 모델 | 분당 요청 | 하루 요청 |
|---|---|---|
| **gemini-3.5-flash-lite (수업용)** | 15 | 500 |
| gemini-3.8-flash (최신 · 고성능) | 5 | 20 |

> 무료 등급의 요청 한도(분당 · 하루)는 **AI Studio 화면에서 확인**합니다. (공식 문서: "View your active rate limits in AI Studio")

In [13]:
# ============================================================
# [실습 5-1] "만약 유료였다면" 계산기
# ------------------------------------------------------------
# 가격은 2026-09-02 구글 발표값(100만 토큰당 달러)입니다.
# 환율은 가정값입니다 — 오늘 환율로 바꿔 넣으세요.
# ============================================================

INPUT_PRICE_PER_MILLION = 0.30     # 달러, 입력 100만 토큰당 (gemini-3.5-flash-lite)
OUTPUT_PRICE_PER_MILLION = 2.50    # 달러, 출력 100만 토큰당 (gemini-3.5-flash-lite)
WON_PER_DOLLAR = 1400              # ★ 가정값 — 오늘 환율로 바꿀 것


def estimate_cost(result):
    """응답의 토큰 수로 유료였다면 낼 금액(원)을 계산한다"""
    input_tokens = result.usage.total_input_tokens or 0
    output_tokens = (result.usage.total_output_tokens or 0) + (result.usage.total_thought_tokens or 0)
    dollars = (input_tokens * INPUT_PRICE_PER_MILLION + output_tokens * OUTPUT_PRICE_PER_MILLION) / 1_000_000
    return dollars * WON_PER_DOLLAR


won = estimate_cost(interaction)
print(f"실습 3-1 요청 한 번 : 약 {won:.4f}원")
print(f"같은 요청 1만 번   : 약 {won * 10_000:,.0f}원")

실습 3-1 요청 한 번 : 약 0.1838원
같은 요청 1만 번   : 약 1,838원


> 생각 토큰도 출력 토큰 가격으로 계산했습니다. 그래서 `thinking_level` 을 높이면 비용도 올라갑니다.

In [14]:
# ============================================================
# [실습 5-2] 한국어와 영어의 토큰 수 비교
# ------------------------------------------------------------
# 같은 뜻의 문장을 보내고 입력 토큰 수를 비교합니다.
# 요금 · 한도는 글자 수가 아니라 토큰 수로 셉니다.
# ============================================================

sentences = {
    "한국어": "오늘 날씨가 정말 좋아서 산책을 다녀왔습니다.",
    "영어": "The weather was really nice today, so I went for a walk.",
}

for language, sentence in sentences.items():
    result = client.interactions.create(
        model=MODEL_NAME,
        input=sentence,
        generation_config={"max_output_tokens": 20},
    )
    print(f"{language:4s} | 글자 {len(sentence):3d}개 | 입력 토큰 {result.usage.total_input_tokens}")

한국어  | 글자  25개 | 입력 토큰 16
영어   | 글자  56개 | 입력 토큰 15


---
# STEP 6. 오류가 나도 멈추지 않게  (약 15분)

API 는 인터넷 너머의 서버를 부르므로 **내 코드가 맞아도 실패할 수 있습니다.** (한도 초과 · 서버 혼잡 · 키 문제)
오류가 나면 `status_code`(상태 번호)로 원인을 구분합니다.

| 상태 번호 | 뜻 | 할 일 |
|---|---|---|
| **429** | 한도 초과 (너무 자주 · 너무 많이 보냄) | 잠시 기다렸다 다시. 하루 한도면 다음 날 |
| **400** | 요청이 잘못됨 (키 오류 · 옵션 이름 오타 등) | 키와 코드 철자 확인 |
| **402** | 결제가 연결된 프로젝트의 키 (선불 잔액 0) | AI Studio 에서 새 프로젝트로 키 다시 발급 |
| **403** | 권한 없음 | 키가 막혔는지 AI Studio 에서 확인 |
| **5xx** | 서버 쪽 문제 | 잠시 뒤 다시 |

> 실습 3-1 에서 `client` 를 만들 때 넣은 `http_options` 덕분에, 429 가 나면 **몇 분씩 말없이 기다리지 않고 바로** 오류가 뜹니다.

In [15]:
# ============================================================
# [실습 6-1] 오류를 잡아 한글로 알려 주는 함수
# ------------------------------------------------------------
# try / except : 실패해도 프로그램이 멈추지 않고 except 안으로 넘어감 (7주차 try/catch 와 같음)
# getattr(error, "status_code", None) : 오류에 상태 번호가 있으면 꺼내고, 없으면 None
# ============================================================

def safe_ask(question):
    """질문을 보내고, 실패하면 원인을 한글로 돌려준다"""
    try:
        result = client.interactions.create(model=MODEL_NAME, input=question)
        return result.output_text
    except Exception as error:
        code = getattr(error, "status_code", None)
        if code == 429:
            return "⏳ 한도를 넘었습니다. 1분 기다렸다 다시 해 보세요."
        if code == 402:
            return "💳 결제가 연결된 프로젝트의 키입니다. AI Studio 에서 새 프로젝트로 키를 다시 만드세요."
        if code == 400:
            return "❌ 요청이 잘못됐습니다. 키와 모델 이름 · 옵션 철자를 확인하세요."
        if code == 403:
            return "❌ 권한이 없습니다. AI Studio 에서 키 상태를 확인하세요."
        return f"❌ 알 수 없는 오류 ({type(error).__name__}, 상태 번호 {code})"


print(safe_ask("안녕? 한 문장으로 인사해 줘."))

안녕하세요! 오늘 어떤 도움이 필요하신가요?


In [16]:
# ============================================================
# [실습 6-2] 일부러 틀려 보기 — 없는 모델 이름
# ------------------------------------------------------------
# 모델 이름을 틀리게 넣으면 어떤 안내가 나오는지 확인합니다.
# ============================================================

def ask_with_model(model_name, question):
    """모델 이름을 바꿔 질문하고, 실패하면 상태 번호를 알려 준다"""
    try:
        result = client.interactions.create(model=model_name, input=question)
        return result.output_text
    except Exception as error:
        return f"❌ 실패 · 상태 번호 {getattr(error, 'status_code', None)} · {type(error).__name__}"


print(ask_with_model("gemini-없는-모델", "안녕?"))

❌ 실패 · 상태 번호 404 · NotFoundError


> 실제 상태 번호와 오류 이름은 실행해서 나온 값을 과제 칸에 적어 두세요.

---
# STEP 7. 무료 등급 보안 · 요금 수칙  (약 10분)

| 수칙 | 이유 (공식 약관 · 문서) |
|---|---|
| **개인정보 · 학번 · 주민번호 · 과제 원본을 넣지 않는다** | "Do not submit sensitive, confidential, or personal information to the Unpaid Services." |
| 무료 등급 입력은 **제품 개선에 쓰일 수 있다** | 무료 등급은 입력과 응답을 구글 제품 개선에 사용하며, 사람이 읽고 검토할 수 있음 |
| **키를 코드 · 깃허브 · 과제 파일에 넣지 않는다** | "Never check API keys into source control systems like Git" |
| 키가 새어 나갔다고 생각되면 **바로 삭제하고 새로 발급** | 발급은 무료. 새 키는 유출 감지 시 빠르게 막힘 |
| **결제 계정을 연결하지 않는다** | 연결하지 않으면 무료 등급에 묶여 요금이 나오지 않음 |
| 만 18세 이상만 사용 | "You must be 18 years of age or older to use the APIs." |
| 무료 등급 대화 기록은 서버에 **1일** 보관 | Interactions API 문서 (유료 등급은 55일) |

> 근거 — Gemini API Additional Terms of Service (2026-04-28 개정) · Using Gemini API keys (2026-09-25) · Interactions API (2026-09-23)

---
# 📝 오늘의 과제 — 나만의 한 줄 도우미

아래 틀을 채워 **특정한 일 하나만 도와주는 함수**를 만듭니다.
(예: 영어 문장 고쳐 주기 · 어려운 낱말 쉽게 풀기 · 요리 재료로 메뉴 추천 · 코드 오류 설명)

### 조건
1. `my_helper(text)` 함수 안에서 `client.interactions.create(...)` 를 부른다
2. 질문 앞에 "무엇을 해 줘" 라는 **지시 문장**을 붙인다 (10주차 프롬프트의 맛보기)
3. 서로 다른 입력 **3개**로 실행하고 결과를 남긴다
4. 각 실행의 **전체 토큰 수**와 **유료였다면 금액**을 함께 출력한다
5. **개인정보를 넣지 않는다**
6. 같은 기능을 **Static Space 웹 페이지**로 올린다 → 아래 **[과제 2단계]**

In [17]:
# ============================================================
# [과제] 나만의 한 줄 도우미
# ------------------------------------------------------------
# (직접 작성) 표시가 있는 곳을 채우세요.
# ============================================================

HELPER_INSTRUCTION = "(직접 작성) 예: 다음 영어 문장의 문법을 고치고, 고친 이유를 한국어 한 줄로 알려 줘."


def my_helper(text):
    """지시 문장 + 입력 글을 보내고 (답 글, 응답 전체) 를 돌려준다"""
    result = client.interactions.create(
        model=MODEL_NAME,
        input=HELPER_INSTRUCTION + "\n\n" + text,
    )
    return result.output_text, result


my_inputs = [
    "(직접 작성) 입력 1",
    "(직접 작성) 입력 2",
    "(직접 작성) 입력 3",
]

for text in my_inputs:
    answer, result = my_helper(text)
    print("입력 :", text)
    print("답   :", answer)
    print(f"토큰 : {result.usage.total_tokens} · 유료였다면 약 {estimate_cost(result):.4f}원")
    print("-" * 40)

입력 : (직접 작성) 입력 1
답   : 요청하신 형식에 맞춰 예시와 입력을 작성해 드립니다.

**[예시]**

*   **원본 문장:** She don't like apples.
*   **고친 문장:** She doesn't like apples.
*   **이유:** 주어가 3인칭 단수('She')이므로 일반동사 부정문에는 'don'이 아니라 'doesn'를 써야 합니다.

---

**[입력 1]**

*   **원본 문장:** I have went to the store yesterday.
*   **고친 문장:** I went to the store yesterday.
*   **이유:** 'yesterday'와 같은 명확한 과거 시점 부사가 있으므로 현재완료(have went)가 아닌 단순과거(went)를 써야 합니다.
토큰 : 220 · 유료였다면 약 0.6437원
----------------------------------------
입력 : (직접 작성) 입력 2
답   : 입력 2가 비어 있습니다. 교정할 영어 문장을 입력해 주시면 요청하신 대로 문법을 고치고 그 이유를 한국어 한 줄로 설명해 드리겠습니다.
토큰 : 82 · 유료였다면 약 0.1607원
----------------------------------------
입력 : (직접 작성) 입력 3
답   : **(직접 작성) 예:**

* **원문:** She don't like apples.
* **수정:** She **doesn't** like apples.
* **이유:** 주어가 3인칭 단수('She')이므로 일반동사의 부정형에는 'don't'가 아니라 'doesn't'를 써야 합니다.

---

**(직접 작성) 입력 3**

* **원문:** Yesterday, I go to the store and buy some milk.
* **수정:** Yesterday, I **went** to the store and **bought** some milk.
* **이유:

---
# 📝 [과제 2단계] 웹으로 올리기 — Static Space

위에서 만든 `my_helper` 와 **같은 일을 하는 웹 페이지**(`index.html`)를 Static Space 에 올립니다. 7주차와 같은 방법입니다.

### ★ API 키는 어떻게 하나
7주차에 배웠듯이 `index.html` 은 **누구나 열어 볼 수 있습니다.** 그래서 키를 코드에 적지 않습니다.
→ 페이지를 쓰는 사람이 **화면의 키 입력칸에 자기 키를 직접 넣습니다.** 키는 그 탭에만 잠시 있고 저장되지 않으며, 탭을 닫으면 사라집니다.

| | 이렇게 하면 | 결과 |
|---|---|---|
| ❌ | `index.html` 안에 `"AIza..."` 처럼 키를 적음 | 누구나 소스 보기로 키를 가져감 |
| ✅ | 화면의 입력칸에 그때그때 키를 넣음 | 코드에는 키가 없음 · 친구가 쓰려면 친구 자신의 키가 필요 |

### 순서
| 순서 | 할 일 |
|---|---|
| 1 | `huggingface.co/spaces` → `Create new Space` → 이름 `my-helper` · SDK **Static** · 템플릿 **Blank** · **Public** |
| 2 | 아래 [과제 2-1] 셀에서 **★ 고칠 곳 ①②③** 을 내 도우미에 맞게 고치고 실행 (③ 에는 위 `HELPER_INSTRUCTION` 과 같은 문장) |
| 3 | [과제 2-2] 셀을 실행 → 찍힌 내용을 **전부 복사** |
| 4 | Space `Files` → `index.html` → `Edit` → **Ctrl + A → Delete → Ctrl + V** → `Commit changes to main` |
| 5 | `App` 탭 → 키 입력칸에 내 키 붙여넣기 → 글을 넣고 **[도와줘]** |
| 6 | 답 · 토큰 · 금액이 나오면 성공 → 주소를 **과제 정리 칸**에 적기 |

### Colab 과제와 나란히 보기
| Colab (파이썬) | 웹 (JavaScript) |
|---|---|
| `HELPER_INSTRUCTION = "…"` | `const HELPER_INSTRUCTION = "…"` (★ 고칠 곳 ③) |
| `client.interactions.create(model=..., input=...)` | `fetch(API_URL, { method: "POST", ... })` — 같은 주소로 요청 |
| `result.output_text` | `getOutputText(data)` — 응답의 `steps` 에서 답 글만 꺼냄 |
| `estimate_cost(result)` | `estimateCost(usage)` |
| `safe_ask` 의 상태 번호 안내 | `explainError(code)` |
| `userdata.get("GEMINI_API_KEY")` | 화면의 키 입력칸 |

> 웹에서는 `google-genai` 같은 도구 없이 **`fetch` 로 구글 서버 주소를 직접** 부릅니다. 보내는 내용(모델 이름 · 질문 · 키)은 파이썬과 같습니다.

In [18]:
%%writefile index.html
<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>영어 문장 고치미</title>

  <style>
    body {
      font-family: "맑은 고딕", sans-serif;
      max-width: 720px;
      margin: 24px auto;
      padding: 0 16px;
      color: #202124;
    }

    h1 {
      font-size: 24px;
      margin-bottom: 4px;
    }

    .sub {
      color: #5f6368;
      margin-top: 0;
      line-height: 1.6;
    }

    .warn {
      background: #fff8e1;
      border: 1px solid #f0d48a;
      border-radius: 6px;
      padding: 8px 12px;
      font-size: 14px;
      line-height: 1.6;
    }

    label {
      display: block;
      font-weight: bold;
      margin: 14px 0 4px;
    }

    input[type="password"],
    textarea {
      width: 100%;
      box-sizing: border-box;
      font-size: 15px;
      padding: 8px;
      border: 1px solid #80868b;
      border-radius: 4px;
    }

    textarea {
      min-height: 100px;
      resize: vertical;
      line-height: 1.6;
    }

    small {
      display: block;
      margin-top: 4px;
      color: #5f6368;
      line-height: 1.5;
    }

    button {
      padding: 8px 16px;
      font-size: 15px;
      margin-top: 8px;
      cursor: pointer;
    }

    button:disabled {
      opacity: 0.5;
      cursor: wait;
    }

    #status {
      display: block;
      margin: 12px 0;
      padding: 8px 12px;
      background: #f1f3f4;
      border-radius: 6px;
      line-height: 1.6;
    }

    h2 {
      font-size: 18px;
      margin: 20px 0 8px;
    }

    #answer {
      padding: 12px;
      min-height: 60px;
      border: 1px solid #dadce0;
      border-radius: 6px;
      font-size: 15px;
      line-height: 1.7;
      white-space: pre-wrap;
      overflow-wrap: anywhere;
    }

    #usage {
      color: #5f6368;
      font-size: 14px;
      margin-top: 8px;
      line-height: 1.6;
      white-space: pre-wrap;
    }
  </style>
</head>

<body>
  <h1>영어 문장 고치미</h1>

  <p class="sub">
    영어 문장의 문법을 고치고, 수정 이유를 한국어 한 문장으로 알려 줍니다.
  </p>

  <p class="warn">
    ⚠ 이름·학번·전화번호 같은 개인정보는 입력하지 마세요.
    무료 등급에 보낸 글은 구글이 제품 개선에 사용할 수 있습니다.
    AI의 수정 결과는 직접 확인하세요.
  </p>

  <label for="keyInput">① 내 Gemini API 키</label>
  <input
    type="password"
    id="keyInput"
    autocomplete="off"
    spellcheck="false"
    placeholder="AI Studio에서 복사한 키를 붙여 넣으세요"
  />
  <small>
    이 페이지는 입력한 키를 별도로 저장하지 않으며,
    API 요청 시 구글 서버로 전송합니다.
  </small>

  <label for="textInput">② 문법을 고칠 영어 문장</label>
  <textarea
    id="textInput"
    placeholder="예: She don't like apples."
    spellcheck="false"
  >She don't like apples.</textarea>
  <small>
    예시: I am go to school every day. / He have two books.
  </small>

  <button id="runButton" type="button">문법 고치기</button>

  <div id="status" role="status" aria-live="polite">
    API 키와 영어 문장을 입력하고 [문법 고치기]를 누르세요.
  </div>

  <h2>문법 수정 결과</h2>
  <div id="answer" aria-live="polite">
    수정한 영어 문장과 한국어 설명이 여기에 표시됩니다.
  </div>
  <div id="usage"></div>

  <script>
    // Python 과제의 HELPER_INSTRUCTION과 같은 문장을 사용합니다.
    const HELPER_INSTRUCTION =
      "다음 영어 문장의 문법을 고쳐 줘. " +
      "'수정 문장:'과 '수정 이유:'로 구분해서 출력하고, " +
      "수정 이유는 한국어 한 문장으로 설명해 줘. " +
      "문법이 맞으면 원문을 그대로 보여 주고 문법이 맞다고 알려 줘.";

    const MODEL_NAME = "gemini-3.5-flash-lite";
    const API_URL =
      "https://generativelanguage.googleapis.com/v1beta/interactions";

    // 수업에서 사용하는 가상 유료 요금 계산용 단가입니다.
    const INPUT_PRICE_PER_MILLION = 0.30;
    const OUTPUT_PRICE_PER_MILLION = 2.50;

    // 실제 환율이 아닌 가정값입니다.
    // 제출 시 확인한 환율로 바꾸고 Python 코드와 맞추세요.
    const WON_PER_DOLLAR = 1400;

    const TIMEOUT_SECONDS = 120;

    const keyInput = document.getElementById("keyInput");
    const textInput = document.getElementById("textInput");
    const runButton = document.getElementById("runButton");
    const statusBox = document.getElementById("status");
    const answerBox = document.getElementById("answer");
    const usageBox = document.getElementById("usage");

    // HTTP 상태 번호에 맞는 안내를 표시합니다.
    function explainError(code) {
      if (code === 400) {
        return "❌ 요청이 잘못되었습니다. API 키, 모델 이름, 요청 옵션을 확인하세요.";
      }

      if (code === 401) {
        return "❌ 인증에 실패했습니다. API 키가 올바른지 확인하세요.";
      }

      if (code === 402) {
        return "❌ 결제 또는 크레딧 관련 오류입니다. 수업용 무료 프로젝트의 키인지 확인하고, 필요하면 교수님께 문의하세요.";
      }

      if (code === 403) {
        return "❌ API 사용 권한이 없습니다. AI Studio에서 키와 프로젝트 상태를 확인하세요.";
      }

      if (code === 404) {
        return "❌ 요청한 모델 또는 API 경로를 찾을 수 없습니다. 모델 이름과 API 주소를 확인하세요.";
      }

      if (code === 429) {
        return "⏳ 요청 한도를 초과했습니다. 잠시 기다린 뒤 다시 시도하세요. 일일 한도라면 분당 한도 대기로 해결되지 않습니다.";
      }

      if (code >= 500) {
        return "❌ 구글 서버에서 오류가 발생했습니다. 잠시 후 다시 시도하세요.";
      }

      return `❌ 요청에 실패했습니다. 상태 번호: ${code}`;
    }

    // 토큰 수로 가상 유료 금액을 계산합니다.
    function estimateCost(usage) {
      const inputTokens = usage.total_input_tokens ?? 0;
      const outputTokens = usage.total_output_tokens ?? 0;
      const thoughtTokens = usage.total_thought_tokens ?? 0;

      const dollars = (
        inputTokens * INPUT_PRICE_PER_MILLION +
        (outputTokens + thoughtTokens) * OUTPUT_PRICE_PER_MILLION
      ) / 1_000_000;

      return dollars * WON_PER_DOLLAR;
    }

    // Interactions 응답의 model_output에서 텍스트를 꺼냅니다.
    function getOutputText(data) {
      if (!Array.isArray(data.steps)) {
        return "";
      }

      return data.steps
        .filter((step) => step.type === "model_output")
        .flatMap((step) =>
          Array.isArray(step.content) ? step.content : []
        )
        .filter((part) =>
          part.type === "text" && typeof part.text === "string"
        )
        .map((part) => part.text)
        .join("\n")
        .trim();
    }

    // AI 답변은 HTML로 실행하지 않고 글자로 표시합니다.
    function showAnswer(text) {
      answerBox.textContent = text;
    }

    async function myHelper() {
      const apiKey = keyInput.value.trim();
      const text = textInput.value.trim();

      if (apiKey === "") {
        statusBox.textContent = "① 내 Gemini API 키를 입력하세요.";
        keyInput.focus();
        return;
      }

      if (text === "") {
        statusBox.textContent = "② 문법을 고칠 영어 문장을 입력하세요.";
        textInput.focus();
        return;
      }

      runButton.disabled = true;
      runButton.textContent = "수정 중...";
      answerBox.textContent = "";
      usageBox.textContent = "";
      statusBox.textContent = "영어 문법을 수정하는 중...";

      const controller = new AbortController();
      const timer = setTimeout(
        () => controller.abort(),
        TIMEOUT_SECONDS * 1000
      );
      const start = performance.now();

      try {
        const response = await fetch(API_URL, {
          method: "POST",
          headers: {
            "Content-Type": "application/json",
            "x-goog-api-key": apiKey
          },
          body: JSON.stringify({
            model: MODEL_NAME,
            input: HELPER_INSTRUCTION + "\n\n" + text
          }),
          signal: controller.signal
        });

        // 응답을 읽는 동안에도 제한 시간을 적용합니다.
        const rawBody = await response.text();

        if (!response.ok) {
          statusBox.textContent = explainError(response.status);
          return;
        }

        let data;
        try {
          data = JSON.parse(rawBody);
        } catch {
          statusBox.textContent =
            "❌ 서버 응답을 해석할 수 없습니다. 잠시 후 다시 시도하세요.";
          return;
        }

        if (!data || typeof data !== "object" || Array.isArray(data)) {
          statusBox.textContent =
            "❌ 예상과 다른 응답 형식입니다. API 응답 형식을 확인하세요.";
          return;
        }

        const answer = getOutputText(data);

        if (answer === "") {
          statusBox.textContent =
            "⚠ 수정 문장이 반환되지 않았습니다. 응답 상태나 안전 제한을 확인하고, 다른 연습 문장으로 시도하세요.";
          return;
        }

        showAnswer(answer);

        const usage = data.usage;
        const seconds = (
          (performance.now() - start) / 1000
        ).toFixed(1);

        if (usage) {
          const hasCostData =
            typeof usage.total_input_tokens === "number" &&
            typeof usage.total_output_tokens === "number";

          const costText = hasCostData
            ? `유료였다면 약 ${estimateCost(usage).toFixed(4)}원`
            : "유료 금액 계산 불가: 토큰 정보가 부족합니다.";

          usageBox.textContent =
            `토큰: 입력 ${usage.total_input_tokens ?? "-"} · ` +
            `출력 ${usage.total_output_tokens ?? "-"} · ` +
            `생각 ${usage.total_thought_tokens ?? 0} · ` +
            `전체 ${usage.total_tokens ?? "-"}\n` +
            `${costText} ` +
            `(계산 환율: 1달러 = ${WON_PER_DOLLAR.toLocaleString()}원)`;
        } else {
          usageBox.textContent =
            "이번 응답에는 토큰 정보가 없어 금액을 계산할 수 없습니다.";
        }

        statusBox.textContent =
          `응답 수신 · ${seconds}초 · 모델: ${MODEL_NAME}` +
          (data.status ? ` · 상태: ${data.status}` : "");

      } catch (error) {
        if (error.name === "AbortError") {
          statusBox.textContent =
            `⏳ ${TIMEOUT_SECONDS}초 안에 응답을 받지 못했습니다. ` +
            "잠시 후 다시 시도하세요.";
        } else {
          statusBox.textContent =
            "❌ 서버에 연결하지 못했습니다. 인터넷 연결과 브라우저의 요청 차단 여부를 확인하세요.";
        }

      } finally {
        clearTimeout(timer);
        runButton.disabled = false;
        runButton.textContent = "문법 고치기";
      }
    }

    runButton.addEventListener("click", myHelper);
  </script>
</body>
</html>

Writing index.html


In [22]:
# ============================================================
# [과제 2-2] index.html 내용을 찍어 복사하기 (또는 내려받기)
# ------------------------------------------------------------
# 찍힌 내용을 전부 복사해 Space 의 index.html 에 붙여 넣습니다.
# 내려받은 파일을 더블클릭해 크롬에서 먼저 시험해 봐도 됩니다.
# ============================================================

print(open("index.html", encoding="utf-8").read())

# 내려받으려면 아래 두 줄 앞의 # 을 지우고 실행하세요
from google.colab import files
files.download("index.html")

<!DOCTYPE html>
<!-- ============================================================
     9주차 과제 · 나만의 한 줄 도우미 (웹 버전 · index.html)
     ------------------------------------------------------------
     Colab 과제의 my_helper(text) 와 같은 일을 브라우저에서 함
     · Static Space 에 그대로 올리면 됨 (서버 · 설치 없음)
     · ★ API 키는 코드에 절대 적지 않음 → 쓰는 사람이 화면에서 직접 입력
       입력한 키는 이 탭의 메모리에만 있고, 탭을 닫으면 사라짐 (어디에도 저장하지 않음)
     ★ 고칠 곳 표시가 붙은 줄만 바꾸면 내 도우미가 됨
     ============================================================ -->
<html lang="ko">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>영어 문장 고치미</title>   <!-- ★ 고칠 곳 ① -->
  <style>
    body { font-family: "맑은 고딕", sans-serif; max-width: 720px; margin: 24px auto; padding: 0 16px; color: #202124; }
    h1 { font-size: 24px; margin-bottom: 4px; }
    .sub { color: #5f6368; margin-top: 0; }
    .warn { background: #fff8e1; border: 1px solid #f0d48a; border-radius: 6px; padding: 8px 12px; font-size: 14

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

> ✅ 확인할 것 — ① Space 의 `Files` 에 키가 적힌 파일이 없는가 ② 화면 캡처 · 발표 때 키 입력칸이 가려져(●●●) 있는가
> ⚠ 친구에게 주소를 줄 때 **내 키를 알려 주지 않습니다.** 친구는 자기 키로 써 봅니다.

### 과제 정리 칸
| 항목 | 내 답 |
|---|---|
| 내 도우미가 하는 일 (한 문장) | |
| 내 Static Space 주소 | https://huggingface.co/spaces/내아이디/my-helper |
| 가장 잘 된 입력과 이유 | |
| 이상하게 나온 입력과 이유 | |
| [실습 6-2] 에서 나온 상태 번호와 오류 이름 | |
| 실제 서비스로 만들 때 걱정되는 점 (한도 · 요금 · 개인정보 중 하나) | |

---
# 🔗 더 알아보기 (2026-09-29 접속 확인)

| 자료 | 주소 |
|---|---|
| Gemini API 빠른 시작 (공식) | https://ai.google.dev/gemini-api/docs/quickstart |
| Interactions API 개요 (공식) | https://ai.google.dev/gemini-api/docs/interactions-overview |
| API 키 사용법 · 보안 (공식) | https://ai.google.dev/gemini-api/docs/api-key |
| 요금 (공식) | https://ai.google.dev/gemini-api/docs/pricing |
| 요청 한도 (공식) | https://ai.google.dev/gemini-api/docs/rate-limits |
| 이용 약관 (공식) | https://ai.google.dev/gemini-api/terms |

### 요즘 이런 일이 있습니다
| 날짜 · 출처 | 내용 | 주소 |
|---|---|---|
| 2026-09-02 · Google 블로그 | Gemini 3.8 Flash 공개 (유료 가격은 2027년에 두 배) | https://blog.google/innovation-and-ai/models-and-research/gemini-models/3-8-flash-and-3-8-flash-cyber/ |
| 2026-06 · Google 공식 문서 | Interactions API 정식 출시, 기존 방식은 legacy | https://ai.google.dev/gemini-api/docs/interactions-overview |
| 2026-09-29 · AI타임스 | 오픈AI, 무료 · 프로토타입 · 가속화 개발자 요금제 준비 | https://www.aitimes.com/news/articleView.html?idxno=215650 |